# PortfolioOS Benchmarked: implementation trade-off research

## 1 — Research question

**What is the trade-off between signal capture, benchmark-relative risk and implementation cost in systematic portfolio construction?**

This is an applied systematic-portfolio-construction question, not a claim of academic novelty.

## 2 — Research mechanism

signal preference → desired active positions → benchmark-risk constraints compress active positions → turnover limits restrict transitions → transaction costs reduce realized net return

This is controlled sensitivity analysis, not causal inference.

## 3 — Data

**SYNTHETIC RESEARCH DEMONSTRATION**

The deterministic default data validate portfolio mechanics. They are not evidence of persistent alpha, and no historical market-performance claim is made. Negative results are retained. The fixed universe contains independent market, sector and idiosyncratic shocks with heterogeneous exposures; no predictive structure is embedded.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

from portfolioos.attribution import sector_attribution, security_attribution
from portfolioos.backtest import run_backtest
from portfolioos.experiments import (
    active_signal_exposure,
    experiment_findings,
    reference_signal_expression_config,
    run_experiment_suite,
    signal_capture,
    write_experiment_report,
)
from portfolioos.reporting import load_config
from portfolioos.synthetic import synthetic_market

root = Path.cwd() if Path("configs/demo.yaml").exists() else Path.cwd().parent
synthetic_settings, config = load_config(root / "configs/demo.yaml")
prices, metadata = synthetic_market(**synthetic_settings)
synthetic_settings

## 4 — Signals

The package combines 12−1 momentum, low volatility and short-term reversal into a standardized cross-sectional preference score. Scores are not expected returns. The backtest calls the same `composite_score` implementation using information only through t−1; the notebook imports it for transparency and does not reimplement it.

## 5 — Reference signal-expression portfolio

The reference uses the same dataset, scores, information dates, benchmark and basic position cap. It remains fully invested and long only. Tracking-error, sector-active and turnover caps, risk aversion and turnover penalty are removed to relax implementation compression. It exists to measure potential signal expression; it is not an optimal alpha portfolio or a more realistic implementation.

In [ ]:
baseline = run_backtest(prices, metadata=metadata, config=config)
reference_config = reference_signal_expression_config(config)
reference = run_backtest(prices, metadata=metadata, config=reference_config)
reference_config.optimizer

## 6 — Signal capture

`ActiveSignalExposure_t = (w_t - b_t)' s_t`

`SignalCapture_t = ActiveSignalExposure_t / ReferenceSignalExposure_t`

Signal capture measures how strongly target active weights align with the chosen score. It does **not** measure alpha, skill, future return, market inefficiency or statistical significance. Near-zero reference exposure produces NaN.

In [ ]:
baseline_capture = signal_capture(baseline, reference)
baseline_exposure = active_signal_exposure(baseline)
baseline_capture.describe(), baseline_exposure.describe()

## 7 — Baseline implementable portfolio

The canonical baseline uses signal weights 1.0/1.0/0.5; an 8% annual TE cap; 8% position cap; 10 percentage-point sector-active cap; 30% one-way turnover cap; 10 bps cost; risk-aversion coefficient 10; and turnover penalty 0.1. Ledoit-Wolf covariance, walk-forward timing, portfolio drift and attribution remain canonical.

In [ ]:
suite = run_experiment_suite(prices, metadata=metadata, base_config=config)
summary = write_experiment_report(suite, root / "results/research_tradeoffs")
summary["dataset_identity"]

## 8 — Tracking-error frontier

The controlled grid is 4%, 6%, 8%, 10% and 12% annual ex-ante TE. The complete table retains signal capture, active exposure, active share, estimated and realized TE, turnover, net active return, information ratio, drawdown and concentration—even when results are unfavorable.

In [ ]:
suite.te_frontier

## 9 — Turnover frontier

The 10%, 20%, 30%, 40% and 50% one-way turnover limits ask: How much signal expression is lost when portfolio transition speed is constrained? Other assumptions stay fixed; tighter turnover is not presumed better or worse.

In [ ]:
suite.turnover_frontier

## 10 — Transaction-cost frontier

The 0, 5, 10, 20 and 40 bps grid is an **accounting sensitivity**. Cost assumptions affect net realized returns; they do not alter historical asset returns, signals or target weights in the current architecture. The optimizer contains a separate turnover penalty rather than an explicit expected-cost model.

In [ ]:
suite.cost_frontier

## 11 — TE × turnover surface

A compact 3 × 3 grid varies only TE and turnover limits to show the implementation surface without an expansive parameter search.

In [ ]:
suite.te_turnover_grid.pivot(
    index="turnover_limit",
    columns="tracking_error_budget",
    values="average_signal_capture",
)

## 12 — Findings

The following statements are generated from the full calculated tables. They are descriptive, retain weak or negative results and make no inference of significance.

In [ ]:
display(Markdown("\n".join(f"- {item}" for item in experiment_findings(suite))))

## 13 — Attribution

The existing engine calculates security active contribution and daily arithmetic Brinson-Fachler allocation, selection and interaction effects. These reconcile to daily gross active return; costs bridge gross to net. Summed daily effects are not compounded multi-period attribution.

In [ ]:
security_attribution(baseline).sum().sort_values().to_frame("contribution")
sector_attribution(baseline).groupby(level="sector").sum()

## 14 — Limitations

The default dataset is synthetic and uses a fixed universe without historical point-in-time membership changes. Execution is idealized at the previous close; costs are simplified and linear; sector metadata are static; constraints are enforced at rebalances and drift can move holdings beyond target limits. Signal capture is not alpha. Controlled sensitivity is not causal evidence. No statistical-significance, institutional-validation or real-world-alpha claim is made.

## 15 — Conclusion

PortfolioOS is a reproducible framework for studying how benchmark-risk and implementation constraints change the expression of systematic signals. The synthetic outcomes assess mechanics; they do not establish that a strategy works in real markets.